In [8]:
import pandas as pd

agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

print(agents.shape)
print(txns.shape)

(20, 3)
(612, 5)


# Mobile Money Agent Data Cleanup and Analysis

# Task 10: Data Quality Audit

This audit identifies issues affecting completeness, validity, consistency, uniqueness and accuracy before cleaning begins.

In [9]:
import pandas as pd
import numpy as np

agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

agents.head()

,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


In [10]:
audit_results = []

# Duplicate transactions
duplicate_count = txns["txn_id"].duplicated().sum()

audit_results.append([
    "Duplicate transaction IDs",
    "Uniqueness",
    "txn_id",
    duplicate_count
])

# Missing amounts
missing_amounts = txns["amount"].isna().sum()

audit_results.append([
    "Missing amounts",
    "Completeness",
    "amount",
    missing_amounts
])

# Invalid transaction types
invalid_types = (
    ~txns["txn_type"]
    .str.strip()
    .str.title()
    .isin(["Deposit", "Withdrawal"])
).sum()

audit_results.append([
    "Inconsistent transaction types",
    "Consistency",
    "txn_type",
    invalid_types
])

# Invalid dates
invalid_dates = (
    txns["txn_date"]
    .astype(str)
    .str.contains("/")
).sum()

audit_results.append([
    "Mixed date formats",
    "Validity",
    "txn_date",
    invalid_dates
])

# Unknown agents
unknown_agents = (
    txns["agent_id"] == "AG099"
).sum()

audit_results.append([
    "Unknown agents",
    "Accuracy",
    "agent_id",
    unknown_agents
])

audit_df = pd.DataFrame(
    audit_results,
    columns=[
        "issue",
        "dimension",
        "column",
        "rows_affected"
    ]
)

audit_df

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing amounts,Completeness,amount,22
2,Inconsistent transaction types,Consistency,txn_type,0
3,Mixed date formats,Validity,txn_date,5
4,Unknown agents,Accuracy,agent_id,8


# Task 11: Duplicate Removal

In [11]:
before_rows = len(txns)

txns = txns.drop_duplicates(
    subset="txn_id"
)

after_rows = len(txns)

duplicates_removed = before_rows - after_rows

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Duplicates removed:", duplicates_removed)

Rows before: 612
Rows after: 600
Duplicates removed: 12


# Task 12: Transaction Type Standardisation


In [12]:
txns["txn_type"] = (
    txns["txn_type"]
    .str.strip()
    .str.title()
)

print(txns["txn_type"].unique())

['Deposit' 'Withdrawal']


# Task 13: Validity and Type Conversion

This task converts transaction amounts to numeric values and transaction dates to datetime format.

In [13]:
txns["amount"] = pd.to_numeric(
    txns["amount"],
    errors="coerce"
)

txns["txn_date"] = pd.to_datetime(
    txns["txn_date"],
    errors="coerce"
)

print(txns.dtypes)

txn_id              object
agent_id            object
txn_type            object
amount             float64
txn_date    datetime64[ns]
dtype: object


# Task 14: Missing Value Treatment

Missing transaction amounts are replaced with the median transaction amount to preserve records while reducing the effect of extreme values.

In [14]:
median_amount = txns["amount"].median()

txns["amount_imputed"] = txns["amount"].isna()

txns["amount"] = txns["amount"].fillna(
    median_amount
)

print("Missing values remaining:")
print(txns["amount"].isna().sum())


Missing values remaining:
0


# Task 15: Outlier Analysis

The IQR method is used to identify unusually large transaction amounts.

In [15]:
Q1 = txns["amount"].quantile(0.25)
Q3 = txns["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

txns["is_outlier"] = (
    (txns["amount"] < lower_bound)
    | (txns["amount"] > upper_bound)
)

print("Outliers Found:")
print(txns["is_outlier"].sum())

Outliers Found:
53


The injected KES 250,000 transactions are likely to be flagged as outliers because they are substantially higher than typical transaction values. These transactions should be investigated by the business before removal because they could represent either genuine large transactions or data-entry errors.

# Task 16: Merge and Validation

Transactions are merged with the agent register using agent_id.

In [16]:
merged = txns.merge(
    agents,
    on="agent_id",
    how="left"
)

unknown_agents = merged["county"].isna().sum()

print("Merged Rows:", len(merged))
print("Unknown Agents:", unknown_agents)

Merged Rows: 600
Unknown Agents: 7


Unknown agents are records whose agent_id does not exist in the agent register. These records were retained during analysis but clearly identified for further investigation.

In [17]:
merged.head()


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,onboarded
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,2024-09-01
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,2025-08-01
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,2024-08-01
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,2024-09-01
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,2024-09-01


# Task 17: County Summary and Weekly Pivot Table

In [20]:
county_summary = merged.groupby(
    ["county", "txn_type"]
).agg(
    total_value=("amount", "sum"),
    transactions=("txn_id", "count"),
    average_value=("amount", "mean")
).round(2)

county_summary

total_value  transactions  average_value
county  txn_type                                            
Kiambu  Deposit        334737.0            43        7784.58
        Withdrawal      77050.5            27        2853.72
Kisumu  Deposit        436420.5            74        5897.57
        Withdrawal      76374.5            37        2064.18
Mombasa Deposit        148986.0            72        2069.25
        Withdrawal     368592.0            61        6042.49
Nairobi Deposit        543590.0           118        4606.69
        Withdrawal     162036.0            62        2613.48
Nakuru  Deposit        184890.0            60        3081.50
        Withdrawal     328594.5            39        8425.50

In [21]:
merged["week"] = merged["txn_date"].dt.isocalendar().week

weekly_pivot = pd.pivot_table(
    merged,
    values="amount",
    index="county",
    columns="week",
    aggfunc="sum"
)

weekly_pivot


week,9,10,11,12,13,14,15,16,17,18
county,,,,,,,,,,
Kiambu,NaN,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,286183.0,23186.0,27592.0,27077.0,39559.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,22242.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0


In [18]:
merged.to_csv(
    "agent_transactions_clean.csv",
    index=False
)

print("Clean file saved successfully")

Clean file saved successfully


In [19]:
from google.colab import files

files.download("agent_transactions_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Data Quality Report

## Completeness

Missing transaction amounts were identified and replaced using the median transaction amount. All imputed values were tracked using the amount_imputed flag.

## Validity

Transaction amounts were converted to numeric values and transaction dates were converted to datetime values to ensure accurate analysis.

## Consistency

Transaction types originally appeared in multiple formats including Deposit, deposit, DEPOSIT and withdrawal. These values were standardized into two valid categories: Deposit and Withdrawal.

## Uniqueness

Duplicate transaction records were identified using txn_id and removed before analysis to ensure each transaction was represented once.

## Accuracy

Transactions linked to agent ID AG099 could not be matched to the agent register and were flagged as unknown agents for further investigation.

## Timeliness

Date formats were standardized to allow accurate weekly summaries and trend analysis.

## Privacy Considerations

If customer phone numbers were included in this dataset, they would qualify as personal data under the Kenya Data Protection Act 2019. Such information should be protected through restricted access, encryption, masking and anonymization where possible.

In [ ]:
from cleaning_pipeline import clean_agent_data

In [ ]:
clean_df, cleaning_log = clean_agent_data(
    txns,
    agents
)

print(cleaning_log)
clean_df.head()